________
generate data

In [132]:
import random
import pandas as pd
from faker import Faker
from datetime import datetime, timedelta

fake = Faker()
Faker.seed(42)
random.seed(42)

def generate_employee_data(num_records=10000):
    employees = []
    cycles = ["Sprint1", "Sprint2", "Sprint3", "Sprint4", "Sprint5"]
    
    for _ in range(num_records):
        year, month = random.randint(2023, 2024), random.randint(1, 12)
        month_str = f"{year}-{str(month).zfill(2)}"
        cycle = random.choice(cycles)
        days = random.choice([5, 10])
        
        # Task Metrics
        assigned_tasks = random.randint(1, 6)
        completed_tasks = random.randint(0, assigned_tasks) 
        task_backlog = assigned_tasks - completed_tasks
        
        task_completion_rate_total = round((completed_tasks / assigned_tasks) * 100, 2) if assigned_tasks > 0 else 0
        # Task Timing
        sprint_start_date = datetime(year, month, random.randint(1, 25))
        task_start_date = sprint_start_date
        task_due_date = sprint_start_date + timedelta(days=days)
        task_ended = task_due_date if completed_tasks > 0 else None

        # Work Hours & Overtime
        minimum_working_hours = 8
        overtime_frequency = random.randint(0, 3)
        if overtime_frequency > 0:
            overtime_hours_per_day = random.randint(1, 3)
        else:
            overtime_hours_per_day = 0
        overtime_hours = overtime_frequency * overtime_hours_per_day
        total_work_hours = (days * minimum_working_hours) + overtime_hours
        avg_working_hours = 40 if days == 5 else 80

        # Task Priority
        num_high_priority_tasks = random.randint(0, assigned_tasks)
        
        # Leave Metrics
        num_leaves_taken = random.randint(0, 5)
        total_leave_days = num_leaves_taken * random.randint(1, 5)
        total_leave_credits = min(total_leave_days, 5)
        
        employees.append([
            month_str, cycle, days, assigned_tasks, completed_tasks, 
            task_backlog, task_completion_rate_total, 
            task_start_date.strftime('%Y-%m-%d'), task_ended.strftime('%Y-%m-%d') if task_ended else None, 
            task_due_date.strftime('%Y-%m-%d'), 
            minimum_working_hours, overtime_frequency, overtime_hours, total_work_hours,avg_working_hours, num_high_priority_tasks, 
            num_leaves_taken, total_leave_days, total_leave_credits
        ])
    
    columns = [
        "month", "cycle", "days", "assigned_tasks", 
        "completed_tasks", "task_backlog", "task_completion_rate_total", 
        "task_start_date", "task_ended", "task_due_date",
        "minimum_working_hours", "overtime_frequency","overtime_hours", "total_work_hours",
        "avg_working_hours", "num_high_priority_tasks", "num_leaves_taken", "total_leave_days", "total_leave_credits"
    ]
    

    df = pd.DataFrame(employees, columns=columns)
    df['burnout_risk'] = df.apply(calculate_burnout_risk, axis=1)
    return df

def calculate_burnout_risk(row):
    risk = 0

    # Total tasks
    if row['assigned_tasks'] >= 5:
        risk += 10

    # Backlog
    if row['task_backlog'] >= 3:
        risk += 10

    # Completion rate (total)
    if row['task_completion_rate_total'] < 60:
        risk += 10

    # Workload intensity
    if (row['days'] == 5 and row['total_work_hours'] > 45) or (row['days'] == 10 and row['total_work_hours'] > 85):
        risk += 10

    # Overtime hours
    if row['overtime_hours'] >= 5:
        risk += 10

    # Leaves taken
    if row['num_leaves_taken'] == 0:
        if row['task_completion_rate_total'] < 75:
            risk += 15

    # High priority tasks ratio
    if row['assigned_tasks'] > 0 and (row['num_high_priority_tasks'] / row['assigned_tasks']) > 0.3:
        risk += 10

    # Ensure risk is within the 0-100 range
    return max(0, min(100, risk))

# Generate data
df = generate_employee_data(15000)
print(df.head())

df.to_csv("synthetic_data.csv", index=False)


     month    cycle  days  assigned_tasks  completed_tasks  task_backlog  \
0  2023-01  Sprint3     5               2                0             2   
1  2023-10  Sprint4     5               1                0             1   
2  2024-04  Sprint4    10               1                0             1   
3  2024-02  Sprint1    10               1                1             0   
4  2024-02  Sprint5    10               6                4             2   

   task_completion_rate_total task_start_date  task_ended task_due_date  \
0                        0.00      2023-01-24        None    2023-01-29   
1                        0.00      2023-10-07        None    2023-10-12   
2                        0.00      2024-04-23        None    2024-05-03   
3                      100.00      2024-02-12  2024-02-22    2024-02-22   
4                       66.67      2024-02-12  2024-02-22    2024-02-22   

   minimum_working_hours  overtime_frequency  overtime_hours  \
0                      8    

In [133]:
# Count unique values in the 'burnout_risk' column
value_counts = df['burnout_risk'].value_counts()

# Print the results
print("\n🔹 Unique Value Counts in 'burnout_risk':")
for value, count in value_counts.items():
    print(f"Value {value} appears {count} times.")



🔹 Unique Value Counts in 'burnout_risk':
Value 10 appears 3457 times.
Value 20 appears 3446 times.
Value 30 appears 2348 times.
Value 40 appears 2107 times.
Value 0 appears 1206 times.
Value 35 appears 410 times.
Value 55 appears 405 times.
Value 50 appears 400 times.
Value 60 appears 397 times.
Value 45 appears 329 times.
Value 25 appears 299 times.
Value 65 appears 100 times.
Value 75 appears 77 times.
Value 15 appears 19 times.


___________________
train

In [134]:
import pandas as pd
import joblib
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression 

In [184]:

# dataset
df = pd.read_csv(r'C:\Users\Sarah\Desktop\try1\ipynb\synthetic_data.csv') 
df.head()


,month,cycle,days,assigned_tasks,completed_tasks,task_backlog,task_completion_rate_total,task_start_date,task_ended,task_due_date,minimum_working_hours,overtime_frequency,overtime_hours,total_work_hours,avg_working_hours,num_high_priority_tasks,num_leaves_taken,total_leave_days,total_leave_credits,burnout_risk
0,2023-01,Sprint3,5,2,0,2,0.00,2023-01-24,NaN,2023-01-29,8,0,0,40,40,2,5,25,5,20
1,2023-10,Sprint4,5,1,0,1,0.00,2023-10-07,NaN,2023-10-12,8,1,3,43,40,0,4,8,5,10
2,2024-04,Sprint4,10,1,0,1,0.00,2024-04-23,NaN,2024-05-03,8,3,6,86,80,1,1,2,2,40
3,2024-02,Sprint1,10,1,1,0,100.00,2024-02-12,2024-02-22,2024-02-22,8,2,2,82,80,1,4,4,4,10
4,2024-02,Sprint5,10,6,4,2,66.67,2024-02-12,2024-02-22,2024-02-22,8,1,3,83,80,0,0,0,0,25


In [185]:
# check missing values
missing_values = df.isnull().sum()
print("Missing Values:\n", missing_values)

Missing Values:
 month                            0
cycle                            0
days                             0
assigned_tasks                   0
completed_tasks                  0
task_backlog                     0
task_completion_rate_total       0
task_start_date                  0
task_ended                    4102
task_due_date                    0
minimum_working_hours            0
overtime_frequency               0
overtime_hours                   0
total_work_hours                 0
avg_working_hours                0
num_high_priority_tasks          0
num_leaves_taken                 0
total_leave_days                 0
total_leave_credits              0
burnout_risk                     0
dtype: int64


In [186]:
#  unused columns
columns_to_exclude = [
    'month','cycle','task_start_date',
    'task_ended','task_due_date'

]
df_model = df.drop(columns=columns_to_exclude, errors='ignore')


In [187]:
#features (X) and target (y)
X = df_model.drop(columns=["burnout_risk"])  # Features
y = df_model["burnout_risk"]  # Target


In [188]:
# 🔹 Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [189]:

# 🔄 Scale numeric features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrame to preserve feature names
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X.columns)


In [190]:
print("\n🔹 Sample of y_test:")
print(y_test[:5])  # Display the first 5 entries 



🔹 Sample of y_test:
11499    30
6475      0
13167    10
862       0
5970     45
Name: burnout_risk, dtype: int64


In [191]:
print("Training Data Shape:", X_train.shape)
print("Testing Data Shape:", X_test.shape)

Training Data Shape: (12000, 14)
Testing Data Shape: (3000, 14)



# Model Training and Evaluation
____________

In [192]:

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train_scaled, y_train)


RandomForestRegressor(random_state=42)

In [194]:

# ✅ Save the trained model
joblib.dump(model, r"C:\Users\Sarah\Desktop\try1\model\rf_regression_model.pkl")

# ✅ Optional: Save the scaler too (so you can use it in Streamlit)
joblib.dump(scaler, r"C:\Users\Sarah\Desktop\try1\model\scaler.pkl")

['C:\\Users\\Sarah\\Desktop\\try1\\model\\scaler.pkl']

In [195]:
obj = joblib.load(r"C:\Users\Sarah\Desktop\try1\model\rf_regression_model.pkl")
print(type(obj))

<class 'sklearn.ensemble._forest.RandomForestRegressor'>


In [196]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# 🔮 Generate predictions first
y_pred = model.predict(X_test_scaled)

# 🔹 Evaluate Random Forest Regressor Performance
print("\n🔹 Random Forest Regressor Performance:")
print("Mean Squared Error:", mean_squared_error(y_test, y_pred))
print("Mean Absolute Error:", mean_absolute_error(y_test, y_pred))
print("R-squared:", r2_score(y_test, y_pred))



🔹 Random Forest Regressor Performance:
Mean Squared Error: 0.09925583333333331
Mean Absolute Error: 0.014850000000000002
R-squared: 0.9996118143217887


In [197]:
# ✅ Comparison DataFrame
comparison_df = pd.DataFrame({
    'Actual': y_test.values,
    'Predicted': y_pred
})

print("\n🔍 Sample Predictions:")
print(comparison_df.head())


🔍 Sample Predictions:
   Actual  Predicted
0      30       30.0
1       0        0.0
2      10       10.0
3       0        0.0
4      45       45.0


____________
Linear 

In [178]:
# importing data 
df = pd.read_csv(r'C:\Users\Sarah\Desktop\try1\ipynb\final_synthetic_data.csv') 
 
print(df.head()) 
print(df.columns)

  first_name last_name emp_id    month    cycle  days  total_tasks  \
0   Danielle   Johnson   E754  2023-01  Sprint3     5            3   
1     Joshua    Walker   E130  2023-04  Sprint2     5            6   
2       Jill    Rhodes   E928  2023-03  Sprint4    10            4   
3   Patricia    Miller   E467  2024-10  Sprint3     5            9   
4     Robert   Johnson   E982  2024-10  Sprint2     5            6   

   assigned_tasks  completed_tasks  task_backlog  ...  minimum_working_hours  \
0               2                2             1  ...                      8   
1               5                5             1  ...                      8   
2               3                1             3  ...                      8   
3               6                4             5  ...                      8   
4               1                0             6  ...                      8   

  overtime_frequency overtime_hours total_work_hours  avg_working_hours  \
0                  0   

In [179]:
#  unused columns
columns_to_exclude = [
    'month','cycle','task_start_date',
    'task_ended','task_due_date'

]
df_model = df.drop(columns=columns_to_exclude, errors='ignore')


In [180]:

#features (X) and target (y)
X = df_model.drop(columns=["burnout_risk"])  # Features
y = df_model["burnout_risk"]  # Target


In [181]:
# creating train and test sets 
X_train, X_test, y_train, y_test = train_test_split( 
    X, y, test_size=0.3, random_state=101) 

In [182]:
# creating a regression model 
model = LinearRegression() 
# fitting the model 
model.fit(X_train,y_train)

ValueError: could not convert string to float: 'Andrew'

In [183]:
# making predictions 
predictions = model.predict(X_test) 

ValueError: could not convert string to float: 'Lauren'

In [154]:
# model evaluation 
print( 
  'mean_squared_error : ', mean_squared_error(y_test, predictions)) 
print( 
  'mean_absolute_error : ', mean_absolute_error(y_test, predictions)) 

mean_squared_error :  55.11732072554784
mean_absolute_error :  5.922310246431641


In [155]:
# ✅ Save the trained model
joblib.dump(model, r"C:\Users\Sarah\Desktop\try1\model\l_regression_model.pkl")


['C:\\Users\\Sarah\\Desktop\\try1\\model\\l_regression_model.pkl']

In [156]:
obj = joblib.load(r"C:\Users\Sarah\Desktop\try1\model\l_regression_model.pkl")
print(type(obj))

<class 'sklearn.linear_model._base.LinearRegression'>


In [157]:
from sklearn.metrics import r2_score

r2 = r2_score(y_test, predictions)
print('R-squared :', r2)


R-squared : 0.7801558030884258
